In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import deque
import random

import environment
from homework2 import Hw2Env

# Hyperparameters
MEMORY_SIZE = 10000
NUM_EPISODES = 2500
BATCH_SIZE = 128
EPS_DECAY = 10000
EPS_END = 0.05
EPS_START = 0.9
GAMMA = 0.99
LEARNING_RATE = 0.0001
TAU = 0.005
N_ACTIONS = 8
STATE_DIM = 6

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32).to(DEVICE),
            torch.tensor(actions, dtype=torch.long).to(DEVICE),
            torch.tensor(rewards, dtype=torch.float32).to(DEVICE),
            torch.tensor(np.array(next_states), dtype=torch.float32).to(DEVICE),
            torch.tensor(dones, dtype=torch.float32).to(DEVICE),
        )

    def __len__(self):
        return len(self.buffer)

# Smoke test
buf = ReplayBuffer(100)
buf.push(np.zeros(6), 0, 1.0, np.ones(6), False)
assert len(buf) == 1
s, a, r, ns, d = buf.sample(1)
assert s.shape == (1, 6)
print("ReplayBuffer OK")

In [ ]:
class DQNNetwork(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x):
        return self.net(x)

# Smoke test
net = DQNNetwork(STATE_DIM, N_ACTIONS).to(DEVICE)
dummy = torch.zeros(1, STATE_DIM).to(DEVICE)
out = net(dummy)
assert out.shape == (1, N_ACTIONS)
print("DQNNetwork OK")

In [ ]:
class DQNAgent:
    def __init__(self):
        self.policy_net = DQNNetwork(STATE_DIM, N_ACTIONS).to(DEVICE)
        self.target_net = DQNNetwork(STATE_DIM, N_ACTIONS).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LEARNING_RATE)
        self.memory = ReplayBuffer(MEMORY_SIZE)
        self.steps_done = 0

    def select_action(self, state):
        eps = EPS_END + (EPS_START - EPS_END) * max(0, (EPS_DECAY - self.steps_done) / EPS_DECAY)
        self.steps_done += 1
        if random.random() < eps:
            return random.randrange(N_ACTIONS)
        with torch.no_grad():
            state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            return self.policy_net(state_t).argmax(dim=1).item()

    def update(self):
        if len(self.memory) < BATCH_SIZE:
            return None
        states, actions, rewards, next_states, dones = self.memory.sample(BATCH_SIZE)

        # Current Q values
        q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Target Q values
        with torch.no_grad():
            next_q = self.target_net(next_states).max(1)[0]
            target_q = rewards + GAMMA * next_q * (1 - dones)

        loss = nn.functional.mse_loss(q_values, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Soft update target network
        for param, target_param in zip(self.policy_net.parameters(), self.target_net.parameters()):
            target_param.data.copy_(TAU * param.data + (1 - TAU) * target_param.data)

        return loss.item()

    def push(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)

# Smoke test
agent = DQNAgent()
dummy_state = np.zeros(6)
action = agent.select_action(dummy_state)
assert 0 <= action < N_ACTIONS
print("DQNAgent OK")

In [ ]:
env = Hw2Env(n_actions=N_ACTIONS, render_mode="offscreen")
agent = DQNAgent()

episode_rewards = []
episode_rps = []

for episode in range(NUM_EPISODES):
    env.reset()
    state = env.high_level_state()
    done = False
    total_reward = 0.0
    steps = 0

    while not done:
        action = agent.select_action(state)
        _, reward, terminal, truncated = env.step(action)
        next_state = env.high_level_state()
        done = terminal or truncated

        agent.push(state, action, reward, next_state, float(done))
        agent.update()

        state = next_state
        total_reward += reward
        steps += 1

    episode_rewards.append(total_reward)
    episode_rps.append(total_reward / steps)

    if (episode + 1) % 100 == 0:
        avg_r = np.mean(episode_rewards[-100:])
        avg_rps = np.mean(episode_rps[-100:])
        eps = EPS_END + (EPS_START - EPS_END) * max(0, (EPS_DECAY - agent.steps_done) / EPS_DECAY)
        print(f"Episode {episode+1}/{NUM_EPISODES} | Avg Reward: {avg_r:.3f} | Avg RPS: {avg_rps:.3f} | Eps: {eps:.3f}")

print("Training complete.")
torch.save(agent.policy_net.state_dict(), 'hw2_policy_run1.pt')
print("Model saved to src/hw2_policy_run1.pt")

In [ ]:
def smooth(data, window=50):
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(episode_rewards, alpha=0.3, color='steelblue', label='Raw')
axes[0].plot(range(49, len(episode_rewards)), smooth(episode_rewards), color='steelblue', label='Smoothed (50)')
axes[0].set_title('Episode Reward')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].legend()

axes[1].plot(episode_rps, alpha=0.3, color='darkorange', label='Raw')
axes[1].plot(range(49, len(episode_rps)), smooth(episode_rps), color='darkorange', label='Smoothed (50)')
axes[1].set_title('Reward Per Step (RPS)')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('RPS')
axes[1].legend()

plt.tight_layout()
plt.savefig('hw2_run1_results.png', dpi=150)
plt.show()
print("Saved hw2_run1_results.png")